In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import (
    pearsonr,
    spearmanr,
    ttest_1samp,
    wilcoxon
)

import statsmodels.formula.api as smf

In [2]:
신설 = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/폐업 및 신설/부산_행정구별_월별_신설법인_202201-202607.csv',
    encoding='utf-8-sig'
)

폐업 = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/폐업 및 신설/21-25전국_부산_폐업수.csv',
    encoding='utf-8-sig'
)

방문 = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/방문자 수/부산시_행정구별_월별_방문자수_통합_202301_202606_v2 복사.csv',
    encoding='utf-8-sig'
)

In [3]:
방문['외부방문자 수'] = (
    방문['외지인 방문자 수']
    + 방문['외국인 방문자 수']
)

방문['연도'] = (
    방문['연월']
    .astype(str)
    .str[:4]
    .astype(int)
)

신설['연도'] = (
    신설['연월']
    .astype(str)
    .str[:4]
    .astype(int)
)

방문_연간 = (
    방문[
        방문['연도'].between(2023, 2025)
    ]
    .groupby(
        ['연도', '행정구'],
        as_index=False
    )
    .agg({
        '외부방문자 수': 'sum',
        '외지인 방문자 수': 'sum',
        '외국인 방문자 수': 'sum',
        '현지인 방문자 수': 'sum',
        '전체 수': 'sum'
    })
)

신설_연간 = (
    신설[
        신설['연도'].between(2023, 2025)
    ]
    .groupby(
        ['연도', '행정구'],
        as_index=False
    )['신설법인수']
    .sum()
)

폐업_연간 = (
    폐업[
        폐업['연도'].between(2023, 2025)
    ]
    .rename(
        columns={
            '지역명': '행정구',
            '법인': '법인폐업수',
            '개인사업자': '개인폐업수',
            '폐업사업자_총계': '전체폐업수'
        }
    )
)

In [4]:
분석 = pd.merge(
    방문_연간,
    폐업_연간[
        [
            '연도',
            '행정구',
            '전체폐업수',
            '법인폐업수',
            '개인폐업수'
        ]
    ],
    on=['연도', '행정구']
)

분석 = pd.merge(
    분석,
    신설_연간,
    on=['연도', '행정구']
)

print(분석.shape)

(48, 11)


In [5]:
def 변화율(col):

    wide = 분석.pivot(
        index='행정구',
        columns='연도',
        values=col
    )

    return (
        wide[2025]
        / wide[2023]
        - 1
    ) * 100

In [6]:
변화 = pd.DataFrame({
    '외부방문증감률':
        변화율('외부방문자 수'),

    '외지인증감률':
        변화율('외지인 방문자 수'),

    '외국인증감률':
        변화율('외국인 방문자 수'),

    '현지인증감률':
        변화율('현지인 방문자 수'),

    '전체폐업증감률':
        변화율('전체폐업수'),

    '개인폐업증감률':
        변화율('개인폐업수'),

    '법인폐업증감률':
        변화율('법인폐업수'),

    '신설증감률':
        변화율('신설법인수')
}).reset_index()

변화

,행정구,외부방문증감률,외지인증감률,외국인증감률,현지인증감률,전체폐업증감률,개인폐업증감률,법인폐업증감률,신설증감률
0,강서구,17.463225,11.472367,106.008803,10.541762,2.529960,1.175779,15.580737,6.754221
1,금정구,4.658071,4.300924,37.625222,-2.822280,-7.681756,-8.982211,12.962963,1.746725
2,기장군,2.849047,1.522218,50.572891,5.004501,-6.678332,-8.375000,17.030568,-3.846154
3,남구,10.332220,8.306552,62.394388,-0.604104,-10.428850,-13.704276,46.846847,1.773050
4,동구,20.660672,18.638337,70.239264,5.368854,2.938517,1.860010,15.976331,-25.203252
5,동래구,16.228442,16.103971,42.881757,7.081875,-2.502980,-2.362403,-5.092593,-0.374532
6,부산진구,8.421381,6.808185,85.245721,0.773653,-2.068868,-2.705546,8.415842,-5.442177
7,북구,6.837478,6.681852,29.976502,1.050763,-6.052783,-6.897590,10.843373,50.318471
8,사상구,7.770037,6.652134,97.388819,-0.534400,-10.033104,-10.106238,-8.984375,-17.940199
9,사하구,10.763174,9.663130,36.830406,5.383357,-7.435397,-8.525937,12.980769,-1.680672


In [7]:
검정대상 = [
    '외부방문증감률',
    '전체폐업증감률',
    '개인폐업증감률',
    '법인폐업증감률',
    '신설증감률'
]

결과 = []

for col in 검정대상:

    values = 변화[col]

    t, p = ttest_1samp(
        values,
        popmean=0
    )

    w, wp = wilcoxon(values)

    결과.append({
        '지표': col,
        '평균증감률': values.mean(),
        't-test p-value': p,
        'Wilcoxon p-value': wp
    })

검정결과 = pd.DataFrame(결과)

검정결과

,지표,평균증감률,t-test p-value,Wilcoxon p-value
0,외부방문증감률,9.678505,0.000002,0.000031
1,전체폐업증감률,-4.615893,0.003001,0.009186
2,개인폐업증감률,-5.547343,0.001312,0.004181
3,법인폐업증감률,9.319256,0.023594,0.018250
4,신설증감률,1.035379,0.876692,0.322510


In [8]:
시각화 = 검정결과.copy()

fig = px.bar(
    시각화,
    x='지표',
    y='평균증감률',
    text='평균증감률',
    title='2023→2025 부산 구·군별 평균 증감률'
)

fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside'
)

fig.add_hline(
    y=0,
    line_dash='dash'
)

fig.update_layout(
    xaxis_title='',
    yaxis_title='평균 증감률 (%)'
)

fig.show()

In [9]:
fig = px.bar(
    변화.sort_values(
        '외부방문증감률',
        ascending=False
    ),
    x='행정구',
    y=[
        '외부방문증감률',
        '전체폐업증감률'
    ],
    barmode='group',
    title='2023→2025 외부방문 증가와 전체폐업 변화'
)

fig.add_hline(
    y=0,
    line_dash='dash'
)

fig.update_layout(
    yaxis_title='증감률 (%)',
    legend_title=''
)

fig.show()

In [10]:
fig = px.bar(
    변화,
    x='행정구',
    y=[
        '개인폐업증감률',
        '법인폐업증감률'
    ],
    barmode='group',
    title='2023→2025 개인사업자와 법인 폐업 변화'
)

fig.add_hline(
    y=0,
    line_dash='dash'
)

fig.update_layout(
    yaxis_title='증감률 (%)',
    legend_title=''
)

fig.show()

In [11]:
# 관광객 증가가 폐업 증가와 관련있는가?

r, p = pearsonr(
    변화['외부방문증감률'],
    변화['전체폐업증감률']
)

print(f'Pearson r = {r:.3f}')
print(f'p-value = {p:.4f}')

# Pearson r = 0.349
# p-value = 0.1858
# 유의하지 않다

Pearson r = 0.349
p-value = 0.1858


In [12]:
fig = px.scatter(
    변화,
    x='외부방문증감률',
    y='전체폐업증감률',
    text='행정구',
    trendline='ols',
    title=(
        '외부방문 증가와 전체폐업 변화'
        f'<br><sup>r={r:.3f}, p={p:.3f}</sup>'
    )
)

fig.add_hline(
    y=0,
    line_dash='dash'
)

fig.add_vline(
    x=0,
    line_dash='dash'
)

fig.update_traces(
    textposition='top center'
)

fig.update_layout(
    xaxis_title='외부방문 증감률 (%)',
    yaxis_title='전체폐업 증감률 (%)'
)

fig.show()